# LeetCode 1882 — Process Tasks Using Servers bằng Python

Link đề: https://leetcode.com/problems/process-tasks-using-servers/ · Độ khó: **Medium** · Chủ đề: Array, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def assignTasks(self, servers: List[int], tasks: List[int]) -> List[int]:
        ...
```

Lộ trình:
1. **Mô phỏng từng giây** đúng như mô tả của đề (hàng đợi task, quét mọi server) — chậm nhưng là "đáp án chuẩn".
2. **Mô phỏng theo task**, quét tuyến tính tìm server rảnh tốt nhất và nhảy thời gian khi cần — `O(m · n)`.
3. **Hai heap**: heap server rảnh `(weight, index)` và heap server bận `(free_time, weight, index)`; nhảy thời gian tới lúc có server rảnh — `O((n + m) log n)`.
4. Biến thể cài đặt: giải phóng server **theo lô** cùng thời điểm và xử lý thời gian bằng `max(t, j)` một cách tường minh.

## 1. Mục tiêu học

1. Đọc kỹ một đề **mô phỏng có nhiều quy tắc ưu tiên** (trọng số nhỏ nhất, chỉ số nhỏ nhất, thứ tự hàng đợi) và chuyển thành **khóa so sánh** của heap.
2. Dùng **hai heap** để mô phỏng hệ thống tài nguyên: một heap theo "độ ưu tiên khi rảnh", một heap theo "thời điểm rảnh trở lại".
3. Xử lý đúng **thời gian**: task `j` vào hàng đợi lúc giây `j`; khi không có server rảnh, **nhảy** tới thời điểm server sớm rảnh nhất (không chờ từng giây).
4. Hiểu vì sao thứ tự gán vẫn đúng khi nhiều server rảnh cùng lúc và nhiều task đang chờ.
5. Kiểm chứng bằng mô phỏng từng giây trên input nhỏ ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho hai mảng (chỉ số từ 0) `servers` (độ dài `n`) và `tasks` (độ dài `m`). `servers[i]` là **trọng số** của server `i`; `tasks[j]` là **thời gian xử lý** (giây) của task `j`.

Các task được phân bằng một **hàng đợi**. Ban đầu mọi server đều rảnh, hàng đợi rỗng.
- Ở giây `j`, task `j` được đưa vào hàng đợi (task 0 ở giây 0).
- Chừng nào còn server rảnh và hàng đợi khác rỗng, task ở **đầu** hàng đợi được gán cho server rảnh có **trọng số nhỏ nhất**; nếu hòa, server có **chỉ số nhỏ nhất**.
- Nếu không có server rảnh mà hàng đợi khác rỗng, ta chờ tới khi một server rảnh rồi gán ngay task tiếp theo. Nếu nhiều server rảnh **cùng lúc**, nhiều task được gán theo thứ tự hàng đợi, tuân theo ưu tiên trọng số/chỉ số ở trên.
- Server nhận task `j` ở giây `t` sẽ rảnh lại ở giây `t + tasks[j]`.

Trả về mảng `ans` độ dài `m`, `ans[j]` là chỉ số server nhận task `j`.

**Ví dụ 1:** `servers = [3,3,2]`, `tasks = [1,2,3,2,1,2]` → `[2,2,0,2,1,2]`.

**Ví dụ 2:** `servers = [5,1,4,3,2]`, `tasks = [2,1,2,4,5,2,1]` → `[1,4,1,4,1,3,2]`.

**Ràng buộc:** `1 ≤ n, m ≤ 2·10⁵`, `1 ≤ servers[i], tasks[j] ≤ 2·10⁵`.

In [1]:
import random
import time
import heapq
from collections import deque
from typing import List

examples = [
    (([3, 3, 2], [1, 2, 3, 2, 1, 2]), [2, 2, 0, 2, 1, 2]),
    (([5, 1, 4, 3, 2], [2, 1, 2, 4, 5, 2, 1]), [1, 4, 1, 4, 1, 3, 2]),
]
for (servers, tasks), want in examples:
    print(f"servers={servers}, tasks={tasks} -> kỳ vọng {want}")

servers=[3, 3, 2], tasks=[1, 2, 3, 2, 1, 2] -> kỳ vọng [2, 2, 0, 2, 1, 2]
servers=[5, 1, 4, 3, 2], tasks=[2, 1, 2, 4, 5, 2, 1] -> kỳ vọng [1, 4, 1, 4, 1, 3, 2]


## 3. Kiến thức nền

### 3.1 `heapq` với tuple làm khóa ưu tiên

Tuple so sánh từng thành phần từ trái sang phải ⇒ `(weight, index)` tự động mã hóa "trọng số nhỏ nhất, hòa thì chỉ số nhỏ nhất".

In [2]:
free = [(3, 0), (3, 1), (2, 2)]
heapq.heapify(free)
print([heapq.heappop(free) for _ in range(3)], "-> thứ tự ưu tiên server")

[(2, 2), (3, 0), (3, 1)] -> thứ tự ưu tiên server


### 3.2 Heap theo thời điểm rảnh

Server bận lưu `(free_time, weight, index)`. Đỉnh heap là server rảnh **sớm nhất**; khi nhiều server rảnh cùng lúc, ta đẩy **tất cả** chúng sang heap rảnh (thứ tự giữa chúng sẽ do heap rảnh quyết định).

In [3]:
busy = [(5, 3, 0), (3, 2, 2), (5, 3, 1)]
heapq.heapify(busy)
t = 5
released = []
while busy and busy[0][0] <= t:
    released.append(heapq.heappop(busy))
print("rảnh ở t=5:", released)

rảnh ở t=5: [(3, 2, 2), (5, 3, 0), (5, 3, 1)]


### 3.3 Hàng đợi FIFO bằng `deque`

Dùng cho mô phỏng từng giây: task `j` vào cuối hàng đợi lúc giây `j`, task đầu hàng được gán trước.

In [4]:
q = deque()
for j in range(3):
    q.append(j)
print(q.popleft(), q)

0 deque([1, 2])


### 3.4 "Nhảy thời gian"

Nếu tại thời điểm `t` không có server rảnh, không cần tăng `t` từng giây — đặt `t = busy[0][0]` (thời điểm rảnh sớm nhất). Đây là khác biệt giữa mô phỏng **theo thời gian** và mô phỏng **theo sự kiện**.

## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 1 `servers = [3,3,2]`, `tasks = [1,2,3,2,1,2]`:

| giây | sự kiện | server rảnh (w, i) | gán |
|---|---|---|---|
| 0 | task 0 vào | (2,2), (3,0), (3,1) | task 0 → server 2 (bận tới 1) |
| 1 | server 2 rảnh; task 1 vào | (2,2), (3,0), (3,1) | task 1 → server 2 (tới 3) |
| 2 | task 2 vào | (3,0), (3,1) | task 2 → server 0 (tới 5) |
| 3 | server 2 rảnh; task 3 vào | (2,2), (3,1) | task 3 → server 2 (tới 5) |
| 4 | task 4 vào | (3,1) | task 4 → server 1 (tới 5) |
| 5 | server 0, 1, 2 rảnh; task 5 vào | (2,2), (3,0), (3,1) | task 5 → server 2 |

⇒ `[2,2,0,2,1,2]`. ✓

**Nhận xét quan trọng:** task được xử lý theo **đúng thứ tự** `0, 1, 2, …` (hàng đợi FIFO), và task `j` không thể bắt đầu trước giây `j`. Vậy thời điểm gán task `j` là `t_j = max(j, thời điểm có server rảnh sau khi các task trước đã gán)`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng giây

**Ý tưởng:** chạy đồng hồ `t = 0, 1, 2, …`. Mỗi giây: (1) giải phóng server có `free_at[i] == t`; (2) nếu `t < m`, đưa task `t` vào hàng đợi; (3) chừng nào còn task chờ và có server rảnh: chọn server rảnh có `(weight, index)` nhỏ nhất bằng **quét tuyến tính**, gán task đầu hàng. Dừng khi đã gán đủ `m` task.

In [5]:
def assign_by_seconds(servers, tasks):
    n, m = len(servers), len(tasks)
    free_at = [0] * n              # server i rảnh từ giây free_at[i]
    q = deque()
    ans = [-1] * m
    done = 0
    t = 0
    while done < m:
        if t < m:
            q.append(t)
        while q:
            best = -1
            for i in range(n):
                if free_at[i] <= t and (best == -1 or (servers[i], i) < (servers[best], best)):
                    best = i
            if best == -1:
                break
            j = q.popleft()
            ans[j] = best
            free_at[best] = t + tasks[j]
            done += 1
        t += 1
    return ans

for (servers, tasks), want in examples:
    print(assign_by_seconds(servers, tasks), "kỳ vọng", want)

[2, 2, 0, 2, 1, 2] kỳ vọng [2, 2, 0, 2, 1, 2]
[1, 4, 1, 4, 1, 3, 2] kỳ vọng [1, 4, 1, 4, 1, 3, 2]


**Độ phức tạp:** số giây mô phỏng `T ≤ m + Σ tasks` (worst khi chỉ có 1 server), mỗi lần gán quét `n` server ⇒ `O(T + m · n)` với `T` có thể tới `4·10¹⁰` — bất khả thi cho ràng buộc thật, nhưng **bám sát từng chữ** của đề.

**Nút thắt:** (1) đi qua từng giây kể cả khi không có sự kiện; (2) quét tuyến tính để tìm server rảnh tốt nhất.

In [6]:
random.seed(0)
for n, m, tmax in [(5, 200, 20), (20, 1000, 50)]:
    s = [random.randint(1, 10) for _ in range(n)]; tk = [random.randint(1, tmax) for _ in range(m)]
    t0 = time.perf_counter(); assign_by_seconds(s, tk)
    print(f"n={n}, m={m}: {time.perf_counter() - t0:.3f}s")

n=5, m=200: 0.000s
n=20, m=1000: 0.001s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (xử lý theo task):** vì task được gán theo thứ tự `0..m−1`, ta duyệt `j` và duy trì `t` = thời điểm hiện tại. Trước khi gán task `j`:
- `t = max(t, j)` (task `j` chưa tới trước giây `j`);
- giải phóng mọi server có `free_time ≤ t`;
- nếu **không** có server rảnh ⇒ `t = thời điểm rảnh sớm nhất` rồi giải phóng mọi server rảnh tại thời điểm đó.

**Insight 2 (hai heap):**
- `free`: `(weight, index)` — đỉnh là server sẽ được chọn.
- `busy`: `(free_time, weight, index)` — đỉnh là server rảnh sớm nhất.

**Invariant:** trước khi gán task `j`, `free` chứa chính xác các server rảnh tại thời điểm `t`, `busy` chứa các server đang bận với thời điểm rảnh `> t`.

**Vì sao "nhảy thời gian" an toàn với nhiều task chờ?** Nếu task `j` phải chờ tới `t' > j`, thì task `j + 1, …` cũng đang chờ (chúng tới lúc `≤ t'` hoặc sau). Gán task `j` tại `t'`, rồi task `j + 1` với `t = max(t', j + 1)` — đúng như hàng đợi FIFO của đề.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Theo task, quét tuyến tính server

Không dùng heap: mỗi task, tính `t = max(t, j)`, tìm server rảnh tốt nhất bằng quét; nếu không có, nhảy `t` tới `min(free_at)` rồi quét lại.

In [7]:
def assign_linear(servers, tasks):
    n = len(servers)
    free_at = [0] * n
    ans = []
    t = 0
    for j, d in enumerate(tasks):
        t = max(t, j)
        best = -1
        for i in range(n):
            if free_at[i] <= t and (best == -1 or (servers[i], i) < (servers[best], best)):
                best = i
        if best == -1:                         # không có server rảnh -> nhảy thời gian
            t = min(free_at)
            for i in range(n):
                if free_at[i] <= t and (best == -1 or (servers[i], i) < (servers[best], best)):
                    best = i
        ans.append(best)
        free_at[best] = t + d
    return ans

for (servers, tasks), want in examples:
    print(assign_linear(servers, tasks), "kỳ vọng", want)

[2, 2, 0, 2, 1, 2] kỳ vọng [2, 2, 0, 2, 1, 2]
[1, 4, 1, 4, 1, 3, 2] kỳ vọng [1, 4, 1, 4, 1, 3, 2]


**Độ phức tạp:** Time `O(m · n)` (tới `4·10¹⁰` worst — vẫn quá chậm), Space `O(n)`. Nhưng đã bỏ được phần "đi từng giây".

### 7.2 Cách 3 — Hai heap

Từng đoạn:
1. `free = [(w, i)]` (heapify), `busy = []`.
2. Với mỗi task `j` (thời gian `d`): `t = max(t, j)`.
3. Chuyển mọi server có `busy[0][0] ≤ t` từ `busy` sang `free`.
4. Nếu `free` rỗng: `t = busy[0][0]`, rồi chuyển mọi server rảnh tại `t`.
5. Pop `(w, i)` từ `free`, `ans[j] = i`, push `(t + d, w, i)` vào `busy`.

In [8]:
def assign_heaps(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]
    heapq.heapify(free)
    busy = []
    ans = [0] * len(tasks)
    t = 0
    for j, d in enumerate(tasks):
        t = max(t, j)
        while busy and busy[0][0] <= t:
            _, w, i = heapq.heappop(busy)
            heapq.heappush(free, (w, i))
        if not free:
            t = busy[0][0]
            while busy and busy[0][0] <= t:
                _, w, i = heapq.heappop(busy)
                heapq.heappush(free, (w, i))
        w, i = heapq.heappop(free)
        ans[j] = i
        heapq.heappush(busy, (t + d, w, i))
    return ans

for (servers, tasks), want in examples:
    print(assign_heaps(servers, tasks), "kỳ vọng", want)

[2, 2, 0, 2, 1, 2] kỳ vọng [2, 2, 0, 2, 1, 2]
[1, 4, 1, 4, 1, 3, 2] kỳ vọng [1, 4, 1, 4, 1, 3, 2]


**Độ phức tạp:** mỗi task gây tối đa một push/pop ở mỗi heap; mỗi server chuyển `busy → free` tối đa một lần cho mỗi lần được gán ⇒ Time `O((n + m) log n)`, Space `O(n + m)` (`ans`).

### 7.3 Cách 4 — Hai heap, không cần giữ `t` toàn cục

Một cách viết khác: nếu `free` rỗng sau khi giải phóng ở thời điểm `j`, **lấy thẳng** server rảnh sớm nhất từ `busy` (không chuyển sang `free`) — nhưng phải cẩn thận: nếu nhiều server rảnh **cùng** thời điểm sớm nhất, chọn theo `(weight, index)`; vì `busy` sắp `(free_time, weight, index)` nên đỉnh `busy` chính là server tốt nhất trong nhóm rảnh sớm nhất. Thời điểm kết thúc là `free_time + d` thay vì `j + d`.

In [9]:
def assign_heaps_v2(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]
    heapq.heapify(free)
    busy = []
    ans = []
    for j, d in enumerate(tasks):
        while busy and busy[0][0] <= j:
            _, w, i = heapq.heappop(busy)
            heapq.heappush(free, (w, i))
        if free:
            w, i = heapq.heappop(free)
            heapq.heappush(busy, (j + d, w, i))
        else:
            ft, w, i = heapq.heappop(busy)          # server rảnh sớm nhất (hòa: w, i nhỏ nhất)
            heapq.heappush(busy, (ft + d, w, i))
        ans.append(i)
    return ans

for (servers, tasks), want in examples:
    print(assign_heaps_v2(servers, tasks), "kỳ vọng", want)

[2, 2, 0, 2, 1, 2] kỳ vọng [2, 2, 0, 2, 1, 2]
[1, 4, 1, 4, 1, 3, 2] kỳ vọng [1, 4, 1, 4, 1, 3, 2]


Vì sao không cần `t` toàn cục? Khi `free` rỗng ở thời điểm `j`, mọi server đều bận, và task `j` sẽ bắt đầu đúng lúc server rảnh sớm nhất `ft` (≥ `j`). Các server khác cũng rảnh ở `ft` vẫn nằm trong `busy` — chúng sẽ được giải phóng ở bước sau (khi `j' ≥ ft`) hoặc được pop trực tiếp theo đúng thứ tự `(ft, w, i)`. Mục 12 kiểm chứng tương đương với Cách 3.

**Độ phức tạp:** như Cách 3.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2 `servers = [5,1,4,3,2]`, `tasks = [2,1,2,4,5,2,1]`:

In [10]:
def trace_heaps(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; ans = []
    for j, d in enumerate(tasks):
        t = max(t, j); rel = []
        while busy and busy[0][0] <= t:
            ft, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i)); rel.append(i)
        jumped = ""
        if not free:
            t = busy[0][0]; jumped = f" (không có server rảnh -> nhảy tới t={t})"
            while busy and busy[0][0] <= t:
                ft, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i)); rel.append(i)
        w, i = heapq.heappop(free); ans.append(i)
        heapq.heappush(busy, (t + d, w, i))
        print(f"task {j} (d={d}) tại t={t}{jumped}: giải phóng {rel or '-'}, gán server {i} (w={w}) "
              f"-> bận tới {t + d}; rảnh còn {sorted(free)}")
    return ans

trace_heaps([5, 1, 4, 3, 2], [2, 1, 2, 4, 5, 2, 1])

task 0 (d=2) tại t=0: giải phóng -, gán server 1 (w=1) -> bận tới 2; rảnh còn [(2, 4), (3, 3), (4, 2), (5, 0)]
task 1 (d=1) tại t=1: giải phóng -, gán server 4 (w=2) -> bận tới 2; rảnh còn [(3, 3), (4, 2), (5, 0)]
task 2 (d=2) tại t=2: giải phóng [1, 4], gán server 1 (w=1) -> bận tới 4; rảnh còn [(2, 4), (3, 3), (4, 2), (5, 0)]
task 3 (d=4) tại t=3: giải phóng -, gán server 4 (w=2) -> bận tới 7; rảnh còn [(3, 3), (4, 2), (5, 0)]
task 4 (d=5) tại t=4: giải phóng [1], gán server 1 (w=1) -> bận tới 9; rảnh còn [(3, 3), (4, 2), (5, 0)]
task 5 (d=2) tại t=5: giải phóng -, gán server 3 (w=3) -> bận tới 7; rảnh còn [(4, 2), (5, 0)]
task 6 (d=1) tại t=6: giải phóng -, gán server 2 (w=4) -> bận tới 7; rảnh còn [(5, 0)]


[1, 4, 1, 4, 1, 3, 2]

Một trace có **nhảy thời gian** (chỉ 2 server, task dài):

In [11]:
trace_heaps([1, 2], [5, 5, 1, 1, 1])

task 0 (d=5) tại t=0: giải phóng -, gán server 0 (w=1) -> bận tới 5; rảnh còn [(2, 1)]
task 1 (d=5) tại t=1: giải phóng -, gán server 1 (w=2) -> bận tới 6; rảnh còn []
task 2 (d=1) tại t=5 (không có server rảnh -> nhảy tới t=5): giải phóng [0], gán server 0 (w=1) -> bận tới 6; rảnh còn []
task 3 (d=1) tại t=6 (không có server rảnh -> nhảy tới t=6): giải phóng [0, 1], gán server 0 (w=1) -> bận tới 7; rảnh còn [(2, 1)]
task 4 (d=1) tại t=6: giải phóng -, gán server 1 (w=2) -> bận tới 7; rảnh còn []


[0, 1, 0, 0, 1]

Bảng tóm tắt trace thứ hai:

| task | t | ghi chú | server |
|---|---|---|---|
| 0 | 0 | rảnh: (1,0),(2,1) | 0 (tới 5) |
| 1 | 1 | rảnh: (2,1) | 1 (tới 6) |
| 2 | 2 → **5** | không có server rảnh, nhảy tới 5 | 0 (tới 6) |
| 3 | 5 → **6** | nhảy tới 6, server 0 và 1 cùng rảnh | 0 (w nhỏ hơn) |
| 4 | 6 | rảnh: (2,1) | 1 |

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def assignTasks(self, servers: List[int], tasks: List[int]) -> List[int]:
        free = [(w, i) for i, w in enumerate(servers)]
        heapq.heapify(free)
        busy = []
        ans = [0] * len(tasks)
        t = 0
        for j, d in enumerate(tasks):
            if t < j:
                t = j
            while busy and busy[0][0] <= t:
                _, w, i = heapq.heappop(busy)
                heapq.heappush(free, (w, i))
            if not free:
                t = busy[0][0]
                while busy and busy[0][0] <= t:
                    _, w, i = heapq.heappop(busy)
                    heapq.heappush(free, (w, i))
            w, i = heapq.heappop(free)
            ans[j] = i
            heapq.heappush(busy, (t + d, w, i))
        return ans

sol = Solution()
for (servers, tasks), want in examples:
    got = sol.assignTasks(servers, tasks)
    print(got, "OK" if got == want else "SAI")

[2, 2, 0, 2, 1, 2] OK
[1, 4, 1, 4, 1, 3, 2] OK


## 10. Vì sao thuật toán đúng

Gọi `s_j` là thời điểm task `j` bắt đầu theo đề. Hàng đợi FIFO + "gán ngay khi có thể" cho:

`s_j = max(j, s_{j−1}, thời điểm sớm nhất ≥ ... có server rảnh sau khi đã gán task 0..j−1)`.

Ta chứng minh bằng quy nạp rằng `t` trong Cách 3 (giá trị lúc gán task `j`) bằng `s_j` và server được chọn đúng.

- **`t ≥ j` và `t ≥ s_{j−1}`:** vì `t` không giảm và được nâng lên `max(t, j)`.
- **Nếu có server rảnh tại `max(s_{j−1}, j)`**: sau khi giải phóng mọi server có `free_time ≤ t`, heap `free` chứa đúng các server rảnh ⇒ task `j` bắt đầu ngay, chọn `(weight, index)` nhỏ nhất — đúng quy tắc.
- **Nếu không có**: đề yêu cầu chờ tới khi **một** server rảnh, tức thời điểm `busy[0][0]` (nhỏ nhất). Tại đó, **mọi** server rảnh cùng lúc đều được giải phóng trước khi chọn, nên khi nhiều server rảnh cùng lúc, task đầu hàng nhận server tốt nhất trong số đó — đúng câu "If multiple servers become free at the same time…".

Task `j + 1` sau đó có `t = max(s_j, j + 1)`: nếu nó đã chờ trong hàng đợi (vì `j + 1 ≤ s_j`), nó được xét ngay tại `s_j` với các server còn rảnh — đúng thứ tự FIFO.

**Cách 4 tương đương** vì khi `free` rỗng, đỉnh của `busy` theo khóa `(free_time, weight, index)` chính là server mà Cách 3 sẽ chọn sau khi nhảy thời gian (server rảnh sớm nhất, hòa thì trọng số/chỉ số nhỏ nhất).

## 11. Phân tích Time / Space

Gọi `n` số server, `m` số task (`≤ 2·10⁵`), `T` = thời điểm task cuối bắt đầu.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Từng giây + quét | `O(T + m n)` | `O(n + m)` | chỉ input nhỏ |
| 2. Theo task + quét | `O(m n)` | `O(n + m)` | không đi từng giây |
| 3. Hai heap | `O((n + m) log n)` | `O(n + m)` | **nộp bài** |
| 4. Hai heap (pop trực tiếp) | `O((n + m) log n)` | `O(n + m)` | ít bước giải phóng |

In [13]:
random.seed(1)
n, m = 2 * 10**5, 2 * 10**5
servers = [random.randint(1, 2 * 10**5) for _ in range(n)]
tasks = [random.randint(1, 2 * 10**5) for _ in range(m)]
for name, fn in [("heaps", assign_heaps), ("heaps_v2", assign_heaps_v2), ("submit", Solution().assignTasks)]:
    t0 = time.perf_counter(); r = fn(servers, tasks)
    print(f"{name:8s}: {time.perf_counter() - t0:.3f}s, ans[:5] = {r[:5]}")

servers_small = [random.randint(1, 10) for _ in range(3)]
tasks_long = [random.randint(1, 2 * 10**5) for _ in range(2 * 10**5)]     # rất ít server -> nhảy thời gian liên tục
t0 = time.perf_counter(); r = Solution().assignTasks(servers_small, tasks_long)
print(f"3 server, 2e5 task dài: {time.perf_counter() - t0:.3f}s")

heaps   : 0.565s, ans[:5] = [32052, 69848, 79786, 64910, 65013]


heaps_v2: 0.533s, ans[:5] = [32052, 69848, 79786, 64910, 65013]


submit  : 0.599s, ans[:5] = [32052, 69848, 79786, 64910, 65013]
3 server, 2e5 task dài: 0.058s


## 12. Bộ test

So mọi cách với mô phỏng từng giây (input nhỏ); trường hợp trọng số bằng nhau, nhiều server rảnh cùng lúc, task dài phải chờ.

In [14]:
fns = [assign_linear, assign_heaps, assign_heaps_v2, lambda s, t: Solution().assignTasks(s, t)]

for (servers, tasks), want in examples:
    assert assign_by_seconds(servers, tasks) == want
    for fn in fns:
        assert fn(servers, tasks) == want

edge = [
    (([1], [1, 1, 1]), [0, 0, 0]),
    (([7, 7, 7], [10, 10, 10, 10]), [0, 1, 2, 0]),            # hòa trọng số -> chỉ số nhỏ nhất
    (([2, 1], [3, 3, 3, 3]), [1, 0, 1, 0]),
    (([1, 2], [5, 5, 1, 1, 1]), None),
]
for (servers, tasks), want in edge:
    ref = assign_by_seconds(servers, tasks)
    if want is not None:
        assert ref == want, (servers, tasks, ref)
    for fn in fns:
        assert fn(servers, tasks) == ref, (fn, servers, tasks)

random.seed(2024)
for _ in range(800):
    n = random.randint(1, 5); m = random.randint(1, 12)
    servers = [random.randint(1, 3) for _ in range(n)]
    tasks = [random.randint(1, 6) for _ in range(m)]
    ref = assign_by_seconds(servers, tasks)
    for fn in fns:
        assert fn(servers, tasks) == ref, (fn, servers, tasks)
print("ví dụ + edge + 800 bộ ngẫu nhiên khớp mô phỏng từng giây")

ví dụ + edge + 800 bộ ngẫu nhiên khớp mô phỏng từng giây


In [15]:
random.seed(77)
for _ in range(30):
    n = random.randint(1, 50); m = random.randint(1, 2000)
    servers = [random.randint(1, 20) for _ in range(n)]
    tasks = [random.randint(1, 300) for _ in range(m)]
    r = assign_linear(servers, tasks)
    assert r == assign_heaps(servers, tasks) == assign_heaps_v2(servers, tasks)
print("input vừa: các cách nhanh khớp Cách 2")

input vừa: các cách nhanh khớp Cách 2


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không nhảy thời gian: dùng `j` làm thời điểm bắt đầu kể cả khi phải chờ

In [16]:
def wrong_start_at_j(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; ans = []
    for j, d in enumerate(tasks):
        while busy and busy[0][0] <= j:
            _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
        if free:
            w, i = heapq.heappop(free)
        else:
            _, w, i = heapq.heappop(busy)
        ans.append(i)
        heapq.heappush(busy, (j + d, w, i))       # SAI: nếu phải chờ, bắt đầu ở thời điểm rảnh, không phải j
    return ans

s, tk = [1, 2], [5, 5, 1, 1, 1]
print("sai =", wrong_start_at_j(s, tk), "| đúng =", assign_by_seconds(s, tk))

sai = [0, 1, 0, 0, 0] | đúng = [0, 1, 0, 0, 1]


Task phải chờ thì kết thúc ở `free_time + d`, không phải `j + d`. **Sửa:** dùng `t` (Cách 3) hoặc `ft + d` (Cách 4).

### Lỗi 2 — Khi nhảy thời gian, chỉ giải phóng **một** server

Nếu nhiều server rảnh cùng thời điểm, phải giải phóng tất cả rồi mới chọn theo `(weight, index)`.

In [17]:
def wrong_release_one(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; ans = []
    for j, d in enumerate(tasks):
        t = max(t, j)
        while busy and busy[0][0] <= t:
            _, i, w = heapq.heappop(busy); heapq.heappush(free, (w, i))
        if not free:
            ft, i, w = heapq.heappop(busy); t = ft       # SAI: chỉ giải phóng MỘT server (chỉ số nhỏ nhất
            heapq.heappush(free, (w, i))                 # trong số rảnh lúc ft), bỏ qua server khác cùng ft
        w, i = heapq.heappop(free); ans.append(i)
        heapq.heappush(busy, (t + d, i, w))              # khóa busy: (thời điểm, chỉ số, trọng số)
    return ans

random.seed(3)
for _ in range(5000):
    n = random.randint(2, 4); m = random.randint(2, 10)
    s = [random.randint(1, 4) for _ in range(n)]; tk = [random.randint(1, 5) for _ in range(m)]
    if wrong_release_one(s, tk) != assign_by_seconds(s, tk):
        print(f"phản ví dụ: servers={s}, tasks={tk}\n  sai  = {wrong_release_one(s, tk)}\n  đúng = {assign_by_seconds(s, tk)}")
        break

phản ví dụ: servers=[3, 4, 4, 3], tasks=[5, 5, 4, 5, 2, 3, 1, 3]
  sai  = [0, 3, 1, 2, 0, 1, 3, 0]
  đúng = [0, 3, 1, 2, 0, 3, 1, 0]


Ở đây heap `busy` dùng khóa `(thời điểm, chỉ số, trọng số)` và khi nhảy thời gian chỉ lấy ra **một** server — server có **chỉ số** nhỏ nhất trong nhóm rảnh cùng lúc, chứ không phải server có **trọng số** nhỏ nhất. (Thú vị: nếu khóa là `(thời điểm, trọng số, chỉ số)` thì lấy một server lại đúng — đó chính là Cách 4. Lỗi nằm ở việc trộn "chỉ giải phóng một" với một khóa không mã hóa đủ quy tắc ưu tiên.) **Sửa:** giải phóng mọi server có `free_time ≤ t` rồi chọn từ `free`.

### Lỗi 3 — Heap `free` chỉ theo trọng số (quên chỉ số khi hòa)

Đẩy `(w, i)` đã tự xử lý hòa. Lỗi xảy ra khi đẩy `(w, ...)` cùng một đối tượng khác không so sánh được, hoặc dùng `(w, -i)`.

In [18]:
def wrong_tie(servers, tasks):
    free = [(w, -i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; ans = []
    for j, d in enumerate(tasks):
        t = max(t, j)
        while busy and busy[0][0] <= t:
            _, w, ni = heapq.heappop(busy); heapq.heappush(free, (w, ni))
        if not free:
            t = busy[0][0]
            while busy and busy[0][0] <= t:
                _, w, ni = heapq.heappop(busy); heapq.heappush(free, (w, ni))
        w, ni = heapq.heappop(free); ans.append(-ni)       # SAI: hòa thì chọn chỉ số LỚN nhất
        heapq.heappush(busy, (t + d, w, ni))
    return ans

s, tk = [7, 7, 7], [10, 10, 10, 10]
print("sai =", wrong_tie(s, tk), "| đúng =", assign_by_seconds(s, tk))

sai = [2, 1, 0, 2] | đúng = [0, 1, 2, 0]


**Sửa:** khóa `(weight, index)` tăng dần.

### Lỗi 4 — Gán task trước khi giải phóng server ở cùng thời điểm

Đề: server rảnh ở giây `t + tasks[j]` có thể nhận task ngay giây đó. Nếu điều kiện giải phóng là `< t` thay vì `≤ t`, server bị coi là còn bận thêm một giây.

In [19]:
def wrong_strict_release(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; ans = []
    for j, d in enumerate(tasks):
        t = max(t, j)
        while busy and busy[0][0] < t:                   # SAI: phải là <=
            _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
        if not free:
            t = busy[0][0]
            while busy and busy[0][0] <= t:
                _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
        w, i = heapq.heappop(free); ans.append(i)
        heapq.heappush(busy, (t + d, w, i))
    return ans

for (servers, tasks), want in examples:
    print("sai =", wrong_strict_release(servers, tasks), "| đúng =", want)

sai = [2, 0, 2, 1, 0, 2] | đúng = [2, 2, 0, 2, 1, 2]
sai = [1, 4, 3, 1, 4, 3, 2] | đúng = [1, 4, 1, 4, 1, 3, 2]


**Sửa:** giải phóng khi `free_time ≤ t`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhiều tài nguyên, chọn tài nguyên rảnh tốt nhất theo nhiều tiêu chí, mỗi tài nguyên bận một khoảng thời gian" ⇒ **hai heap**: rảnh (theo ưu tiên) + bận (theo thời điểm rảnh).
- Tiêu chí ưu tiên nhiều tầng ⇒ **tuple** làm khóa heap.
- Mô phỏng theo **sự kiện**: nhảy thời gian tới sự kiện kế tiếp thay vì đi từng đơn vị.
- Cẩn thận `≤` vs `<` ở thời điểm giải phóng.
- Họ bài: 1834 *Single-Threaded CPU*, 2402 *Meeting Rooms III* (gần y hệt), 253 *Meeting Rooms II*, 1701 *Average Waiting Time*.

## 15. Biến thể của bài

1. **Meeting Rooms III (2402):** phòng chọn theo chỉ số nhỏ nhất, cuộc họp chờ thì bị **lùi** giữ nguyên độ dài — cùng khung hai heap; trả về phòng tổ chức nhiều nhất.
2. **Trả về thời điểm bắt đầu/kết thúc của mỗi task** — lưu thêm `t` khi gán.
3. **Task đến ở thời điểm tùy ý** (mảng `arrival` không giảm) thay vì giây `j`: thay `t = max(t, j)` bằng `t = max(t, arrival[j])`.
4. **Server có tốc độ** (thời gian xử lý = `tasks[j] / speed[i]`): khóa heap `busy` dùng số thực / phân số.

In [20]:
def meeting_rooms_iii(n, meetings):
    """Biến thể 1 (LeetCode 2402): trả về phòng tổ chức nhiều cuộc họp nhất (hòa: chỉ số nhỏ nhất)."""
    meetings = sorted(meetings)
    free = list(range(n)); heapq.heapify(free); busy = []; cnt = [0] * n
    for s, e in meetings:
        while busy and busy[0][0] <= s:
            _, r = heapq.heappop(busy); heapq.heappush(free, r)
        if free:
            r = heapq.heappop(free); heapq.heappush(busy, (e, r))
        else:
            ft, r = heapq.heappop(busy); heapq.heappush(busy, (ft + e - s, r))
        cnt[r] += 1
    return cnt.index(max(cnt))

def assign_with_times(servers, tasks, arrival=None):
    """Biến thể 2 + 3: trả về (server, start, end) cho mỗi task; arrival mặc định là j."""
    arrival = arrival or list(range(len(tasks)))
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; out = []
    for j, d in enumerate(tasks):
        t = max(t, arrival[j])
        while busy and busy[0][0] <= t:
            _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
        if not free:
            t = busy[0][0]
            while busy and busy[0][0] <= t:
                _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
        w, i = heapq.heappop(free); out.append((i, t, t + d))
        heapq.heappush(busy, (t + d, w, i))
    return out

print("2402 ví dụ:", meeting_rooms_iii(2, [[0, 10], [1, 5], [2, 7], [3, 4]]), meeting_rooms_iii(3, [[1, 20], [2, 10], [3, 5], [4, 9], [6, 8]]))
print("ví dụ 1 có thời điểm:", assign_with_times([3, 3, 2], [1, 2, 3, 2, 1, 2]))

2402 ví dụ: 0 1
ví dụ 1 có thời điểm: [(2, 0, 1), (2, 1, 3), (0, 2, 5), (2, 3, 5), (1, 4, 5), (2, 5, 7)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Từng giây | mô phỏng nguyên văn | `O(T + mn)` | `O(n + m)` | kiểm chứng |
| 2. Theo task + quét | nhảy thời gian, quét server | `O(mn)` | `O(n + m)` | `n` rất nhỏ |
| 3. Hai heap | rảnh/bận | `O((n+m) log n)` | `O(n + m)` | **nộp bài** |
| 4. Hai heap, pop trực tiếp | không giữ `t` | `O((n+m) log n)` | `O(n + m)` | gọn hơn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `assignTasks([1], [3, 1, 1])` và `assignTasks([2, 1, 2], [1, 1, 1, 1])`?

**Bài 2 (tính tay):** lập bảng sự kiện cho `servers = [10, 20]`, `tasks = [4, 1, 1, 3]` (thời điểm, server rảnh, gán).

**Bài 3 (điền TODO):** hoàn thiện `finish_time(servers, tasks)` trả về thời điểm task **cuối cùng** hoàn thành.

**Bài 4 (tự viết lại):** viết lại Cách 3 sao cho mọi thao tác giải phóng server nằm trong **một** hàm lồng `release(upto)` (tránh lặp code hai vòng `while`).

**Bài 5 (mini challenge):** biến thể 3 — task `j` đến ở thời điểm `arrival[j]` (không giảm). Kiểm chứng `assign_with_times` bằng mô phỏng từng giây có `arrival`. (Biến thể tự đặt.)

In [21]:
def finish_time(servers, tasks):
    # TODO: chạy mô phỏng hai heap, trả về max(t + d) của mọi task
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Một server: mọi task vào server 0. (2) `[0, 0, 0]`. (3) Ba server `[2,1,2]`, task dài 1: mỗi giây server 1 (w=1) rảnh lại ⇒ `[1, 1, 1, 1]`.

**Bài 2:** (1) `t=0`: task 0 (4s) → server 0 (w=10), bận tới 4. (2) `t=1`: task 1 → server 1, tới 2; `t=2`: server 1 rảnh, task 2 → server 1 tới 3. (3) `t=3`: server 1 rảnh, task 3 → server 1 ⇒ `[0, 1, 1, 1]`.

**Bài 3:** (1) Dùng `assign_with_times`. (2) Lấy `max(end)`. (3) Kiểm tra với mô phỏng từng giây.

**Bài 4:** (1) `def release(upto)`: chuyển mọi server `busy[0][0] ≤ upto`. (2) Gọi `release(t)`, nếu `free` rỗng: `t = busy[0][0]; release(t)`. (3) Kết quả phải khớp Cách 3.

**Bài 5:** (1) Mô phỏng từng giây: task `j` vào hàng đợi khi `t == arrival[j]` (có thể nhiều task cùng giây). (2) So sánh `server` của `assign_with_times`. (3) Sinh `arrival` không giảm ngẫu nhiên.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert Solution().assignTasks([1], [3, 1, 1]) == [0, 0, 0]
assert Solution().assignTasks([2, 1, 2], [1, 1, 1, 1]) == [1, 1, 1, 1]

# Bài 2
assert Solution().assignTasks([10, 20], [4, 1, 1, 3]) == [0, 1, 1, 1] == assign_by_seconds([10, 20], [4, 1, 1, 3])

# Bài 3
def finish_time_ans(servers, tasks):
    return max(end for _, _, end in assign_with_times(servers, tasks))

# Bài 4
def assign_release_fn(servers, tasks):
    free = [(w, i) for i, w in enumerate(servers)]; heapq.heapify(free); busy = []; t = 0; ans = []
    def release(upto):
        while busy and busy[0][0] <= upto:
            _, w, i = heapq.heappop(busy); heapq.heappush(free, (w, i))
    for j, d in enumerate(tasks):
        t = max(t, j); release(t)
        if not free:
            t = busy[0][0]; release(t)
        w, i = heapq.heappop(free); ans.append(i); heapq.heappush(busy, (t + d, w, i))
    return ans

def finish_by_seconds(servers, tasks):
    n, m = len(servers), len(tasks); free_at = [0] * n; q = deque(); done = 0; t = 0; last = 0
    while done < m:
        if t < m: q.append(t)
        while q:
            best = min((i for i in range(n) if free_at[i] <= t), key=lambda i: (servers[i], i), default=-1)
            if best == -1: break
            j = q.popleft(); free_at[best] = t + tasks[j]; last = max(last, t + tasks[j]); done += 1
        t += 1
    return last

random.seed(171)
for _ in range(500):
    s = [random.randint(1, 3) for _ in range(random.randint(1, 4))]; tk = [random.randint(1, 5) for _ in range(random.randint(1, 10))]
    assert assign_release_fn(s, tk) == assign_by_seconds(s, tk)
    assert finish_time_ans(s, tk) == finish_by_seconds(s, tk)
print("Bài 1-4: OK")

Bài 1-4: OK


In [23]:
# Bài 5 — arrival tùy ý (không giảm)
def assign_by_seconds_arrival(servers, tasks, arrival):
    n, m = len(servers), len(tasks); free_at = [0] * n; q = deque(); ans = [-1] * m; done = 0; t = 0; nxt = 0
    while done < m:
        while nxt < m and arrival[nxt] == t:
            q.append(nxt); nxt += 1
        while q:
            best = min((i for i in range(n) if free_at[i] <= t), key=lambda i: (servers[i], i), default=-1)
            if best == -1: break
            j = q.popleft(); ans[j] = best; free_at[best] = t + tasks[j]; done += 1
        t += 1
    return ans

random.seed(172)
for _ in range(600):
    n = random.randint(1, 4); m = random.randint(1, 10)
    s = [random.randint(1, 3) for _ in range(n)]; tk = [random.randint(1, 5) for _ in range(m)]
    arr = sorted(random.randint(0, 12) for _ in range(m))
    got = [i for i, _, _ in assign_with_times(s, tk, arr)]
    assert got == assign_by_seconds_arrival(s, tk, arr), (s, tk, arr)
print("Bài 5: biến thể arrival tùy ý khớp mô phỏng từng giây")

Bài 5: biến thể arrival tùy ý khớp mô phỏng từng giây


## 20. Template nhớ nhanh + Checklist

```python
free = [(w, i) for i, w in enumerate(servers)]; heapify(free)   # (trọng số, chỉ số)
busy = []                                                        # (thời điểm rảnh, trọng số, chỉ số)
t = 0
for j, d in enumerate(tasks):
    t = max(t, j)                                 # task j chưa tới trước giây j
    while busy and busy[0][0] <= t: release()     # giải phóng mọi server rảnh tới t
    if not free:
        t = busy[0][0]                            # nhảy tới lúc rảnh sớm nhất
        while busy and busy[0][0] <= t: release() # giải phóng TẤT CẢ server rảnh cùng lúc
    w, i = heappop(free); ans[j] = i
    heappush(busy, (t + d, w, i))
```

**Checklist trước khi submit:**
- [ ] Khóa heap rảnh `(weight, index)`; khóa heap bận `(free_time, weight, index)`.
- [ ] `t = max(t, j)` — không bắt đầu task trước giây `j`, không lùi thời gian.
- [ ] Khi chờ: thời điểm kết thúc là `t_rảnh + d`, không phải `j + d`.
- [ ] Giải phóng khi `free_time ≤ t` (không phải `<`).
- [ ] Giải phóng **mọi** server rảnh cùng thời điểm trước khi chọn.